# Template matching notebook for tilt series and selected rois

This notebook can be run after preprocessing the data. Regions of interest (rois) are chosen from different crystals. The subsets are then template matched, in order to index the crystals from the different tilts.

## Contents
 1. [Load the data](#1.-Load-the-data)
 2. [Define/load rois](#2.-Define/load-rois)
 3. [Create subsets](#3.-Create-subsets)
 4. [Polar transform](#4.-Polar-transform)
 5. [Create a library of simulated diffraction patterns](#5.-Create-library-of-simulated-diffraction-patterns)
 6. [Load simulations](#7.-Load-simulations)
 7. [Template match](#7.-Template-match)
 8. [Save results](#8.-Save-results)
 9. [axis-angle-selected orientations](#9.-Create-crystal-maps)
 10. [IPF validation plots](#10.-Save-representative-pixels)
 11. [Save](#1.-Save)


goal
1. Finalize representative_pixels.json
2. Make one clean m-3m IPF plot for A–D
3. Make one stereographic plot with all arcs A–D
4. Make misorientation-vs-tilt plot
5. Save all figures
6. Write 5–8 sentences explaining limitations

In [1]:
%matplotlib widget

import json
import pickle
from pathlib import Path
import requests

import warnings
warnings.filterwarnings('ignore')

import hyperspy.api as hs
import pyxem as pxm
import numpy as np
import matplotlib.pyplot as plt #Plotting tools
import matplotlib.colors as mcolors #Some plotting color tools
from matplotlib.cm import ScalarMappable
import diffpy #Electron diffraction tools
from matplotlib.colors import to_rgb


#Import indexation and plotting tools
from diffsims.generators.rotation_list_generators import get_beam_directions_grid
from diffsims.libraries.structure_library import StructureLibrary
from diffsims.generators.diffraction_generator import DiffractionGenerator
from diffsims.generators.library_generator import DiffractionLibraryGenerator
from diffsims.generators.simulation_generator import SimulationGenerator

#Import orientation handling tools
from orix.quaternion import Rotation, symmetry, Orientation, OrientationRegion
from orix.crystal_map import Phase
from orix.vector.vector3d import Vector3d
from orix.vector import Miller
from orix.projections import StereographicProjection
from orix import plot
from orix.plot import IPFColorKeyTSL
from orix.sampling import get_sample_reduced_fundamental
from orix.io import save, load

from sklearn.decomposition import PCA
import itertools

In [2]:
#import functions from src
import sys
import importlib

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import src.roi_utils
import src.template_matching
importlib.reload(src.template_matching)
import src.axis_angle
importlib.reload(src.axis_angle)

from src.roi_utils import load_rois, save_rois_by_tilt, extract_roi_subsets
from src.template_matching import load_polar_rois, run_template_matching

from src.axis_angle import get_candidates_for_pixel, select_candidates_axis_angle
from src.axis_angle import run_axis_angle_selection

# 1. Load the data

In [3]:
# load data from 7 tilts
datasets = [
    {"tilt": +10, "path": Path("../../data/20260417_Ingrid/093932/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_18p2_Ty3p8_cropped_centered_masked.hspy")},
    {"tilt": +5,  "path": Path("../../data/20260417_Ingrid/100104/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_13p2_Ty3p8_cropped_centered_masked.hspy")},
    {"tilt": 0,   "path": Path("../../data/20260417_Ingrid/091336/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_8p2_Ty3p8_big_centered_masked.hspy")},
    {"tilt": -5,  "path": Path("../../data/20260417_Ingrid/101949/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_3p2_Ty3p8_cropped_centered_masked.hspy")},
    {"tilt": -10, "path": Path("../../data/20260417_Ingrid/103611/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_m2p2_Ty3p8_cropped_centered_masked.hspy")},
    {"tilt": -15, "path": Path("../../data/20260417_Ingrid/105201/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_m7p2_Ty3p8_cropped_centered_masked.hspy")},
    {"tilt": -20, "path": Path("../../data/20260417_Ingrid/110843/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_m12p2_Ty3p8_cropped_centered_masked.hspy")},
]
results_path = Path("../results")

In the next cell, the SPED datasets are loaded lazily using HyperSpy in order to reduce memory usage during preprocessing, in addition to be chunked and normalized.

Each diffraction pattern is normalized to its maximum intensity before template matching: I_norm = I/(I_max + e), where e = 10^(-8) avoids division by zero. This reduces intensity variations between diffraction patterns and imporves robustness of template matching across tilt series.



In [4]:
lazy = True

loaded = {}

for d in datasets:
    
    tilt = d["tilt"]
    datapath = d["path"]

    print(f"Loading tilt {tilt}")

    signal = hs.load(str(datapath), lazy=lazy)

    if lazy:
        signal.rechunk(
            nav_chunks=(32, 32),
            sig_chunks=(32, 32)
        )
    
    # normalize each diffraction pattern
    signal.map(lambda x: x / (x.max() + 1e-8), inplace=True)

    print(f"  Shape: {signal.data.shape}  dtype: {signal.data.dtype}")
    
    loaded[tilt] = signal

Loading tilt 10
  Shape: (109, 187, 256, 256)  dtype: float64
Loading tilt 5
  Shape: (164, 229, 256, 256)  dtype: float64
Loading tilt 0
  Shape: (153, 209, 256, 256)  dtype: float64
Loading tilt -5
  Shape: (151, 232, 256, 256)  dtype: float64
Loading tilt -10
  Shape: (194, 234, 256, 256)  dtype: float64
Loading tilt -15
  Shape: (124, 198, 256, 256)  dtype: float64
Loading tilt -20
  Shape: (163, 235, 256, 256)  dtype: float64


# 2. Define/load rois

In [5]:
signals = loaded

roi_path = results_path / "roi_by_tilt.json"

if roi_path.exists():
        
        print(f"Loading ROIs")
        roi_by_tilt = load_rois(roi_path)

else:
    
    print("No ROI file. Define interactively")

    roi_by_tilt = {}

    for tilt, signal in signals.items():

        print(f"\n Tilt {tilt}")
        

        roi_A = hs.roi.RectangularROI(left=100.58, right=177.99, top=603.77, bottom=635.18)
        roi_B = hs.roi.RectangularROI(left=247.79, right=279.20, top=572.36, bottom=603.77)
        roi_C = hs.roi.RectangularROI(left=474.64, right=506.05, top=607.26, bottom=638.67)
        roi_D = hs.roi.RectangularROI(left=628.20, right=659.61, top=771.29, bottom=602.70)

        signal.plot(norm="symlog")

        roi_A.add_widget(signal, color="red")
        roi_B.add_widget(signal, color="aqua")
        roi_C.add_widget(signal, color="mediumorchid")
        roi_D.add_widget(signal, color="hotpink")

        roi_by_tilt[tilt] = {
            "A": roi_A,
            "B": roi_B,
            "C": roi_C,
            "D": roi_D
        }

Loading ROIs


## Save and visualize new rois

In [ ]:
assert False

In [ ]:
# save current ROIs
save_rois_by_tilt(roi_by_tilt, roi_path)

In [ ]:
colors = {
    "A": "red",
    "B": "aqua",
    "C": "mediumorchid",
    "D": "hotpink"
}

for tilt, signal in signals.items():

    print(f"Tilt {tilt}")

    # plot dataset
    signal.plot(norm="symlog")

    rois = roi_by_tilt[tilt]

    # add overlays
    for key, roi in rois.items():
        roi.add_widget(signal, color=colors[key])

# 3. Create subsets

In [ ]:
roi_subsets = extract_roi_subsets(
    signals,
    roi_by_tilt
)

In [ ]:
#plot rois from one tilt to inspect
tilt = 0  # choose your reference tilt

for i, key in enumerate(["A", "B", "C", "D"]):

    subset = roi_subsets[tilt][key]

    subset.plot(norm="symlog", cmap="magma")

plt.tight_layout()

# 4. Polar transform

In [ ]:
roi_polar = {}

npt = 128
npt_azim=360

polar_root = results_path / "polar_rois"
polar_root.mkdir(parents=True, exist_ok=True)

for tilt, roi_dict in roi_subsets.items():

    print(f"\nProcessing tilt {tilt}")

    roi_polar[tilt] = {}

    tilt_dir = polar_root / f"tilt_{tilt}"
    tilt_dir.mkdir(parents=True, exist_ok=True)

    for key, subset in roi_dict.items():

        print(f"  ROI {key}")

        #prepare data

        subset = subset.copy()
        subset.change_dtype("float32")

        #convert form cartesian to polar coordinates
        subset_pol = subset.get_azimuthal_integral2d(
            npt=npt,
            npt_azim=npt_azim
        )

        # save polar data
        roi_polar[tilt][key] = subset_pol

        subset_pol.save(tilt_dir / f"ROI_{key}_polar.hspy")


    

# 5. Create library of simlulated diffraction patterns

create simulations

In [ ]:
fname = 'GaAs.cif'
url = "https://materials.springer.com/downloads/track-required/true?path=%2Fassets%2Fsm_isp%2Fcrystallographic%2F311%2Fsm_isp_sd_0311662%2Fsm_isp_sd_0311662_download.cif&componentId=Download+Data+CIF"
r = requests.get(url)
open(fname , 'wb').write(r.content)

GaAs = Phase.from_cif(fname)

In [ ]:
GaAs = Phase.from_cif("../../GaAs.cif")

angular_resolution = 0.5
grid = get_sample_reduced_fundamental(
    resolution=angular_resolution,
    point_group=GaAs.point_group
)

sim_gen = SimulationGenerator(
    accelerating_voltage=200,
    precession_angle=1,
    minimum_intensity=1e-10
)

simulations = sim_gen.calculate_diffraction2d(
    phase=GaAs,
    rotation=grid,
    reciprocal_radius=2,
    with_direct_beam=False
)

orientations = Orientation(
    grid,
    symmetry=GaAs.point_group
)

key_x = IPFColorKeyTSL(GaAs.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(GaAs.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(GaAs.point_group, Vector3d.zvector())

In [9]:
simulation_root = results_path / "simulation_library"
simulation_root.mkdir(parents=True, exist_ok=True)

sim_path = simulation_root / "GaAs_simulations_angres0p5_r2_mi1em10.pkl"
grid_path = simulation_root / "GaAs_grid_angres0p5.pkl"

In [ ]:
simulation_root = results_path / "simulation_library"
simulation_root.mkdir(parents=True, exist_ok=True)

sim_path = simulation_root / "GaAs_simulations_angres0p5_rr2.pkl"
grid_path = simulation_root / "GaAs_grid_angres0p5.pkl"

with open(sim_path, "wb") as f:
    pickle.dump(simulations, f)

with open(grid_path, "wb") as f:
    pickle.dump(grid, f)

print("Simulation library saved.")

In [ ]:
#visualize orienations
orientations.scatter('ipf')
orientations

In [ ]:
simulations.plot(interactive=True)

# 5. Load saved simulation

In [16]:
GaAs = Phase.from_cif("../../GaAs.cif")

with open(sim_path, "rb") as f:
    simulations = pickle.load(f)

with open(grid_path, "rb") as f:
    grid = pickle.load(f)

orientations = Orientation(
    grid,
    symmetry=GaAs.point_group,
)

key_x = IPFColorKeyTSL(GaAs.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(GaAs.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(GaAs.point_group, Vector3d.zvector())

print("Simulation library loaded.")

Simulation library loaded.


# 6. Template match

For each ROI and each tilt, the polar-transformed diffraction patterns are compared with the simulated diffraction library.

The best-matching simulated crystal orientaitons are returned for each scan pixel.

Workflow:
diffraction pattern -> polar transform -> simulation comparison -> best-matching orientation

## one

In [ ]:
tilt = -15
key = "C"

n_keep = 8
npt = 128
npt_azim = 360

subset = roi_subsets[tilt][key].copy()

subset.change_dtype("float32")

subset_pol = subset.get_azimuthal_integral2d(
    npt=npt,
    npt_azim=npt_azim
)

res = subset_pol.get_orientation(
    simulation=simulations,
    n_best=8,
    frac_keep=1.0
)

subset_pol.plot(norm="symlog")

subset_pol.add_marker(
    res.to_single_phase_polar_markers(
        signal_axes=subset_pol.axes_manager.signal_axes
    )
)

In [ ]:
subset = roi_subsets[-15]["C"]
subset.plot(cmap='viridis_r', norm='symlog')
subset.add_marker(res.to_markers(annotate=True))


In [ ]:
roi_results = {}
roi_results[tilt] = {}
res = subset_pol.get_orientation(
            simulation=simulations,
            n_best=n_keep,
            frac_keep=1.0
        )

        roi_results[tilt][key] = res

In [ ]:
assert False

### check one match

In [ ]:
pol_0_A = roi_polar[-15]["C"]


pol_0_A.plot()

pol_0_A.add_marker(roi_results[-15]["C"].to_single_phase_polar_markers(signal_axes=pol_0_A.axes_manager.signal_axes))


In [ ]:
res_0_A= pol_0_A.get_orientation(simulations, n_best=n_keep, frac_keep=1.0)
res_0_A



In [ ]:
res_0_A = roi_subsets[-15]["C"]

res_0_A.plot(cmap='viridis_r', norm='log')
res_0_A.add_marker(roi_results[-15]["C"].to_markers(annotate=True))


# Load polar-transformed ROIs

In [11]:
polar_root = results_path / "polar_rois"

roi_polar = load_polar_rois(polar_root)


Tilt -10
Found 4 files
Loading ROI A
Loading ROI B
Loading ROI C
Loading ROI D
complete

Tilt -15
Found 4 files
Loading ROI A
Loading ROI B
Loading ROI C
Loading ROI D
complete

Tilt -20
Found 4 files
Loading ROI A
Loading ROI B
Loading ROI C
Loading ROI D
complete

Tilt -5
Found 4 files
Loading ROI A
Loading ROI B
Loading ROI C
Loading ROI D
complete

Tilt 0
Found 4 files
Loading ROI A
Loading ROI B
Loading ROI C
Loading ROI D
complete

Tilt 10
Found 4 files
Loading ROI A
Loading ROI B
Loading ROI C
Loading ROI D
complete

Tilt 5
Found 4 files
Loading ROI A
Loading ROI B
Loading ROI C
Loading ROI D
complete


## Compute (template match)

In [17]:
roi_results = run_template_matching(
    roi_polar,
    simulations,
    n_keep=8,
)

Tilt -20, ROI A
WARNING | Hyperspy | The function you applied does not take into account the difference of scales in-between axes. (hyperspy.signal:5597)
WARNING | Hyperspy | The function you applied does not take into account the difference of units in-between axes. (hyperspy.signal:5602)
TM complete
Tilt -20, ROI B
WARNING | Hyperspy | The function you applied does not take into account the difference of scales in-between axes. (hyperspy.signal:5597)
WARNING | Hyperspy | The function you applied does not take into account the difference of units in-between axes. (hyperspy.signal:5602)
TM complete
Tilt -20, ROI C
WARNING | Hyperspy | The function you applied does not take into account the difference of scales in-between axes. (hyperspy.signal:5597)
WARNING | Hyperspy | The function you applied does not take into account the difference of units in-between axes. (hyperspy.signal:5602)
TM complete
Tilt -20, ROI D
WARNING | Hyperspy | The function you applied does not take into account th

In [ ]:
# quick inspection
tilt = -20
roi_key = "B"

pol = roi_polar[tilt][roi_key]
res = roi_results[tilt][roi_key]

pol.plot()
pol.add_marker(
    res.to_single_phase_polar_markers(
        signal_axes=pol.axes_manager.signal_axes
    )
)

## Candidate selection and quality control

For each ROI, a representative scan pixel was chosen based on the IPF-Z map and correlation score map.  
The highest-scoring template-matching candidate at this pixel was used for the main stereographic analysis.

An axis-angle trajectory check was additionally tested to identify possible branch jumps in the tilt series. This method assumes that the orientation evolves smoothly with tilt. However, because this criterion can occasionally select crystallographically inconsistent candidates, it was used only as a diagnostic tool.

# Extract best candidates

For each ROI, the center pixel is used.
All n_best candidates are collected across tilts.
Candidate 0 defines the initial trajectory.
Outliers from a PCA line in axis-angle space are replaced by the candidate closest to that trajectory.

Best main method:
high-CI ROI medoid orientation

QC:
IPF-Z maps, CI maps, stereographic continuity

Diagnostic:
axis-angle trajectory check

## 1. IPF-Z plots
sanity check for template matching plotting the highest scoring pixel.

direction = Vector3d.zvector(). which is the beam direction / zone axis direction

plot one twin across all tilts

In [ ]:
twin_key = "C"

tilt_keys = sorted(roi_results.keys())  # assumes roi_results[tilt]
n = len(tilt_keys)

fig = plt.figure(figsize=(2*n, 4))

for i, tilt in enumerate(tilt_keys):

    res = roi_results[tilt][twin_key]

    # ori shape: nav_x, nav_y, n_best

    oris = res.to_single_phase_orientations()[:, 0]
    ori = res.to_single_phase_orientations()[7, 0, 0]
    

    ax = fig.add_subplot(1, n, i + 1, projection="ipf", symmetry=GaAs.point_group)

    ax.scatter(
        ori,
        cmap="inferno",
    )

    ax.set_title(f"\nTilt {tilt} | Twin {twin_key}\n")

plt.tight_layout()
plt.show()

In [ ]:
# Plot one twin across all tilts
twin_key = "B"
tilt_keys = np.array(sorted(roi_results.keys()))

all_oris = []
all_tilts = []

for tilt in tilt_keys:
    res = roi_results[tilt][twin_key]
    ori = res.to_single_phase_orientations()[6, 7, 0]

    all_oris.append(ori)
    all_tilts.append(tilt)

oris_all = Orientation(
    np.stack([o.data.squeeze() for o in all_oris]),
    symmetry=GaAs.point_group
)

tilt_colors = np.array(all_tilts)

cmap = plt.cm.plasma
norm = plt.Normalize(vmin=tilt_keys.min(), vmax=tilt_keys.max())
rgba = cmap(norm(tilt_colors))

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="ipf", symmetry=GaAs.point_group)

ax.scatter(
    oris_all,
    color=rgba,
    alpha=0.9,
    s=80
)

ax.set_title(f"Twin {twin_key}\n")

sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, shrink=0.9)
cbar.set_label("Tilt")
cbar.set_ticks(tilt_keys)
cbar.set_ticklabels([str(t) for t in tilt_keys])

plt.tight_layout()
plt.show()

In [ ]:
# Plot one twin across all tilts
twin_key = "B"
tilt_keys = np.array(sorted(roi_results.keys()))

# data
all_oris = []
all_tilts = []

for tilt in tilt_keys:

    res = roi_results[tilt][twin_key]

    oris = res.to_single_phase_orientations()[0, 0, 0]

    all_oris.append(oris)

    n = oris.data.shape[0]
    all_tilts.append(np.full(n, tilt))

oris_all = Orientation(
    np.concatenate([o.data for o in all_oris]),
    symmetry=symmetry.Oh
)

tilt_colors = np.concatenate(all_tilts)

# colormap
cmap = plt.cm.plasma
norm = plt.Normalize(
    vmin=tilt_keys.min(),
    vmax=tilt_keys.max()
)

rgba = cmap(norm(tilt_colors))

# ---- plot ----
fig = plt.figure(figsize=(5, 4))

ax = fig.add_subplot(
    111,
    projection="ipf",
    symmetry=symmetry.Oh
)

ax.scatter(
    oris_all,
    color=rgba,
    alpha=0.9,
    s=80
)

ax.set_title(f"Twin {twin_key}")

# colorbar
sm = plt.cm.ScalarMappable(
    norm=norm,
    cmap=cmap
)

sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    shrink=0.8
)

cbar.set_label("Tilt")
cbar.set_ticks(tilt_keys)
cbar.set_ticklabels([str(t) for t in tilt_keys])

plt.tight_layout()
plt.show()

m-3m

# Extract best candidates with axis-angle

##
Template matching occasionally produces symmetry-equivalent orientation branches.
Axis-angle continuity was therefore explored as a method for identifying branch jumps.
However, crystallographic consistency in IPF-Z space and stereographic trajectories was used as the final selection criterion.

Extracting best candidate inspired by: https://github.com/denislobe/Masteroppgave/blob/main/Code/aluminium_new.ipynb

For each twin:
1. pick a representative pixel
2. collect all best candidates at that pixel for all tilts
3. start with candidate 0
4. fit a smooth tilt trajectory in axis-angle space
5. find tilts where candidate 0 is off the trajectory
6. replace only those bad tilts with a better candidate
7. store the corrected orientation sequence

using axis-angle space, where any crystal orientation can be represented as `rotation axis + rotation angle`, for example 60° around [111]. This then becomes a vector: `rotation_vector = axis * angle`.
The tilt series represent "smooth" orientation changes -> smooth curve in 3D axis-angle space. One could represent the crystal orientations in Euler angles, IPF positions and stereographic coordinates, but points in a 3D mathematical orientaion space, as axis-angle space is, can better show tilt orientations in a line.




In [18]:
axis_angle_results = run_axis_angle_selection(
    roi_results,
    roi_keys=["A", "B", "C", "D"],
    symmetry=GaAs.point_group,
    threshold_percentile=70,
)

TypeError: run_axis_angle_selection() got an unexpected keyword argument 'roi_keys'

In [ ]:
for twin_key in ["A", "B", "C", "D"]:

    r = axis_angle_results[twin_key]

    tilt_keys = r["tilts"]
    center_oris = r["center_oris"]
    axis_oris = r["axis_oris"]

    cmap = plt.cm.plasma
    norm = plt.Normalize(vmin=tilt_keys.min(), vmax=tilt_keys.max())
    rgba = cmap(norm(tilt_keys))

    fig = plt.figure(figsize=(10, 4))

    ax1 = fig.add_subplot(121, projection="ipf", symmetry=symmetry.Oh)
    ax1.scatter(center_oris, color=rgba, s=90, alpha=0.9)
    ax1.set_title(f"Twin {twin_key}: candidate 0 \n")

    ax2 = fig.add_subplot(122, projection="ipf", symmetry=symmetry.Oh)
    ax2.scatter(axis_oris, color=rgba, s=90, alpha=0.9)
    ax2.set_title(f"Twin {twin_key}: axis-angle selected \n")

    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = fig.colorbar(sm, ax=[ax1, ax2], shrink=0.85)
    cbar.set_label("Tilt")
    cbar.set_ticks(tilt_keys)

    plt.show()

## save axis-angle orientations

In [ ]:
with open(results_path / "axis_angle_selected_orientations.pkl", "wb") as f:
    pickle.dump(axis_angle_results, f)

print("Axis-angle selected orientations saved.")

# IPF plots

In [ ]:
# Plot axis-angle selected orientations
# one row per twin, one IPF panel per tilt

twin_keys = ["A", "B", "C", "D"]

for twin_key in twin_keys:

    r = axis_angle_results[twin_key]

    tilt_keys = r["tilts"]
    axis_oris = r["axis_oris"]
    chosen_idx = r["chosen_idx"]

    n = len(tilt_keys)

    #cmap = plt.cm.plasma
    norm = plt.Normalize(
        vmin=tilt_keys.min(),
        vmax=tilt_keys.max()
    )

    fig = plt.figure(figsize=(2*n, 4))

    for i, tilt in enumerate(tilt_keys):

        ori = axis_oris[i]

        #rgba = cmap(norm(tilt))

        ax = fig.add_subplot(
            1,
            n,
            i + 1,
            projection="ipf",
            symmetry=GaAs.point_group
        )

        ax.scatter(
            ori,
            s=100
        )

        ax.set_title(
            f"Tilt {tilt} \n",
            fontsize = 14
        )

    fig.suptitle(
        f"Twin {twin_key}: axis-angle selected orientations \n",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()

test

In [19]:
twin_key = "D"

r = axis_angle_results[twin_key]
px, py = r["center_pixel"]

for test_tilt in [-5, 0]:

    print(f"\nTwin {twin_key}, tilt {test_tilt}")

    res = roi_results[test_tilt][twin_key]
    ori_full = res.to_single_phase_orientations()

    for idx in range(8):

        ori = ori_full[px, py, idx]

        v = ori * Vector3d.zvector()

        print(f"candidate {idx}: vector = {v.data.squeeze()}")

NameError: name 'axis_angle_results' is not defined

In [ ]:
for idx_m5 in range(8):
    for idx_0 in range(8):

        ori_m5 = roi_results[-5]["D"].to_single_phase_orientations()[px, py, idx_m5]
        ori_0  = roi_results[0]["D"].to_single_phase_orientations()[px, py, idx_0]

        v_m5 = ori_m5 * Vector3d.zvector()
        v_0  = ori_0  * Vector3d.zvector()

        ang = float(v_m5.angle_with(v_0, degrees=True))

        if ang < 10:
            print(
                f"-5 cand {idx_m5} → 0 cand {idx_0}: "
                f"{ang:.2f}°"
            )

## stereograpchic

In [ ]:
def get_twin_vectors_selected(twin_key):

    tilts = selected_orientations[twin_key]["tilts"]
    oris = selected_orientations[twin_key]["orientations"]

    vectors = oris * Vector3d.zvector()

    return tilts, vectors


def fit_arc_pole(vectors):

    X = vectors.data.reshape(-1, 3)
    X = X / np.linalg.norm(X, axis=1)[:, None]

    _, _, vh = np.linalg.svd(X, full_matrices=False)

    axis_vec = vh[-1]
    axis_vec /= np.linalg.norm(axis_vec)

    return Vector3d(axis_vec)
x

# -------------------------
# load Denis-selected vectors
# -------------------------
tilts, vA = get_twin_vectors_selected("A")
tilts, vB = get_twin_vectors_selected("B")

# -------------------------
# fit stereographic arcs
# -------------------------
arc_pole_A = fit_arc_pole(vA)
arc_pole_B = fit_arc_pole(vB)

# -------------------------
# stereographic plot
# -------------------------
fig = vA.scatter(
    c="red",
    s=90,
    axes_labels=["RD", "TD", None],
    show_hemisphere_label=True,
    return_figure=True,
)

vB.scatter(
    figure=fig,
    c="aqua",
    s=90
)

# -------------------------
# arc poles
# -------------------------
arc_pole_A.scatter(
    figure=fig,
    c="red",
    marker="*",
    s=250
)

arc_pole_B.scatter(
    figure=fig,
    c="blue",
    marker="*",
    s=250
)

# -------------------------
# fitted great circles
# -------------------------
arc_pole_A.draw_circle(
    figure=fig,
    color="red",
    linewidth=2
)

arc_pole_B.draw_circle(
    figure=fig,
    color="blue",
    linewidth=2
)

# -------------------------
# formatting
# -------------------------
ax = fig.axes[0]

ax.set_title(
    "Twins A and B:"
)

ax.stereographic_grid(True)

fig

In [ ]:
angle_AB = float(
    arc_pole_A.angle_with(
        arc_pole_B,
        degrees=True
    )
)

angle_AB_small = min(angle_AB, 180 - angle_AB)

print(f"A-B arc-center angle: {angle_AB:.2f}°")
print(f"A-B smallest equivalent angle: {angle_AB_small:.2f}°")

### misorientatin

In [ ]:
# -------------------------
# A-B misorientation from selected orientations
# -------------------------
oris_A = selected_orientations["A"]["orientations"]
oris_B = selected_orientations["B"]["orientations"]
tilts = selected_orientations["A"]["tilts"]

print("A-B misorientation per tilt")

mis_AB = []

for i, tilt in enumerate(tilts):

    mis = float(
        oris_A[i].angle_with(
            oris_B[i],
            degrees=True
        )
    )

    mis_AB.append(mis)

    print(f"Tilt {tilt:>3}: {mis:.2f}°")

print(f"\nMean A-B misorientation: {np.mean(mis_AB):.2f}°")
print(f"Std: {np.std(mis_AB):.2f}°")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(
    tilts,
    mis_AB,
    marker="o",
    label="A-B"
)

ax.axhline(
    60,
    color="k",
    linestyle="--",
    label="Expected twin 60°"
)

ax.set_xlabel("Tilt")
ax.set_ylabel("Misorientation (°)")
ax.set_title("A-B twin misorientation vs tilt")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()

## test

In [ ]:
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from orix.quaternion import Orientation, OrientationRegion
from orix.vector import Vector3d

In [ ]:
# -------------------------
# 1. Collect all candidates
# -------------------------
twin_key = "C"
px, py = 6, 0

tilt_keys = np.array(sorted(roi_results.keys()))

all_candidates = []

for tilt in tilt_keys:
    res = roi_results[tilt][twin_key]
    ori_full = res.to_single_phase_orientations()

    # all n_best candidates at same pixel
    all_candidates.append(ori_full[px, py])

candidate_oris = Orientation(
    np.stack([o.data for o in all_candidates]),
    symmetry=GaAs.point_group
)

print(candidate_oris.shape)   # should be (n_tilts, n_best)

In [ ]:
def axang_to_xyz(axang):
    xyz = np.array(axang.xyz)

    # want shape (n_points, 3)
    if xyz.shape[0] == 3 and xyz.ndim == 2:
        xyz = xyz.T

    xyz = np.squeeze(xyz)

    if xyz.shape[-1] != 3:
        raise ValueError(f"Unexpected axis-angle xyz shape: {xyz.shape}")

    return xyz

In [ ]:
candidate0 = candidate_oris[:, 0]

axang = candidate0.to_axes_angles()
xyz = axang_to_xyz(axang)

print("xyz shape:", xyz.shape)

pca = PCA(n_components=1)
pca.fit(xyz)

centroid = xyz.mean(axis=0)

direction = pca.components_[0]
direction = direction / np.linalg.norm(direction)

diff = xyz - centroid
dist = np.linalg.norm(np.cross(diff, direction), axis=1)

threshold = np.percentile(dist, 70)
inlier_mask = dist < threshold

print("Distances:", dist)
print("Inliers:", inlier_mask)

In [ ]:
fixed_oris = Orientation(
    candidate_oris[:, 0].data.copy(),
    symmetry=GaAs.point_group
)

chosen_idx = np.zeros(len(tilt_keys), dtype=int)

for i, tilt in enumerate(tilt_keys):

    if inlier_mask[i]:
        chosen_idx[i] = 0
        continue

    candidates = candidate_oris[i]

    cand_ax = candidates.to_axes_angles()
    cand_xyz = axang_to_xyz(cand_ax)

    diff = cand_xyz - centroid
    cand_dist = np.linalg.norm(np.cross(diff, direction), axis=1)

    best_idx = int(np.argmin(cand_dist))

    fixed_oris[i] = candidates[best_idx]
    chosen_idx[i] = best_idx

for tilt, idx in zip(tilt_keys, chosen_idx):
    print(f"Tilt {tilt:>3}: chosen candidate {idx}")

In [ ]:


# -------------------------
# m-3m IPF plot
# -------------------------

ipf_symmetry = symmetry.Oh   # m-3m

cmap = plt.cm.plasma

norm = plt.Normalize(
    vmin=tilt_keys.min(),
    vmax=tilt_keys.max()
)

fig = plt.figure(figsize=(5, 4))

ax = fig.add_subplot(
    111,
    projection="ipf",
    symmetry=ipf_symmetry
)

sc = ax.scatter(
    fixed_oris,
    c=tilt_keys,
    alpha=0.9,
    cmap=cmap,
    norm=norm,
    s=80
)

ax.set_title(
    f"Twin {twin_key}"
)

# -------------------------
# colorbar
# -------------------------
sm = mpl.cm.ScalarMappable(
    norm=norm,
    cmap=cmap
)

sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    shrink=0.9
)

cbar.set_label("Tilt")

cbar.set_ticks(tilt_keys)
cbar.set_ticklabels([str(t) for t in tilt_keys])

plt.tight_layout()
plt.show()

## check bad candidate

In [ ]:
twin_key = "B"
bad_tilt = -20

# choose the candidate that visually lies on the correct IPF branch
manual_fix = {
    -20: 0,   # change to 2,3,4... after checking
}

fixed_oris = selected_orientations[twin_key]["orientations"]
candidate_oris = selected_orientations[twin_key]["candidates"]
tilt_keys = selected_orientations[twin_key]["tilts"]
chosen_idx = selected_orientations[twin_key]["chosen_candidate"]

for i, tilt in enumerate(tilt_keys):
    if tilt in manual_fix:
        idx = manual_fix[tilt]
        fixed_oris[i] = candidate_oris[i, idx]
        chosen_idx[i] = idx

selected_orientations[twin_key]["orientations"] = fixed_oris
selected_orientations[twin_key]["chosen_candidate"] = chosen_idx

### use xmap to check bad tilt

In [ ]:
tilt = -20
roi_key = "B"

signal_results = roi_results[tilt][roi_key]

# make non-lazy copy
signal_results_c = signal_results.deepcopy()
signal_results_c.compute()

xmap = signal_results_c.to_crystal_map()

xmap.prop["index"] = signal_results_c.isig[0, 0].data.flatten()
xmap.prop["ci"] = signal_results_c.isig[1, 0].data.flatten()
xmap.prop["rotation"] = signal_results_c.isig[2, 0].data.flatten()
xmap.prop["mirrored"] = signal_results_c.isig[3, 0].data.flatten()

In [ ]:
plt.figure(figsize=(5, 4))
plt.imshow(xmap.ci.reshape(xmap.shape), cmap="magma")
plt.colorbar(label="CI")
plt.scatter(4, 4, c="cyan", marker="x", s=120, label="[4,4]")
plt.legend()
plt.title("ROI C, tilt -20: confidence map")
plt.show()

In [ ]:
key_z = IPFColorKeyTSL(
    GaAs.point_group,
    Vector3d.zvector()
)

# RGB colors from orientations
rgb = key_z.orientation2color(xmap.orientations)

# reshape to image
rgb_map = rgb.reshape(xmap.shape + (3,))
#  mark candidate pixels on C -20 IPF-Z map
plt.figure(figsize=(5, 4))
plt.imshow(rgb_map)
#plt.scatter(4, 4, c="white", marker="x", s=150, label="[4,4]")
plt.scatter(0, 0, c="black", marker="x", s=150, label="[6,7]")
plt.legend()
plt.title("Twin C, tilt -20: IPF-Z candidate pixels")
plt.show()

In [ ]:
key_z = IPFColorKeyTSL(GaAs.point_group, Vector3d.zvector())

fig = xmap.plot(
    key_z.orientation2color(xmap.orientations),
    remove_padding=True,
    return_figure=True,
    scalebar=False
)

fig.suptitle("ROI C, tilt -20: IPF-Z map")

## old compute

In [ ]:
# choose pixel in the mid of the roi and store all candidates at this pixel
roi_results = {}
roi_scores = {}
roi_orientations = {}

n_keep = 8

for tilt, roi_dict in roi_polar.items():

    roi_results[tilt] = {}
    roi_scores[tilt] = {}
    roi_orientations[tilt] = {}

    for roi_key, signal_pol in roi_dict.items():

        print(f"Tilt {tilt}, ROI {roi_key}")

        res = signal_pol.get_orientation(
            simulation=simulations,
            n_best=n_keep,
            frac_keep=1.0
        )

        roi_results[tilt][roi_key] = res

        data = res.data
        scores = data[..., 0, 1]

        ori_full = res.to_single_phase_orientations()

        cx = ori_full.shape[0] // 2
        cy = ori_full.shape[1] // 2

        # all n_best candidates at center pixel
        candidates_raw = ori_full[cx, cy]

        candidates = Orientation(
            candidates_raw.data,
            symmetry=GaAs.point_group
        ).map_into_symmetry_reduced_zone()

        roi_scores[tilt][roi_key] = {
            "mean": float(scores.mean()),
            "max": float(scores.max()),
            "std": float(scores.std()),
            "center_scores": data[cx, cy, :, 1].compute()
                if hasattr(data[cx, cy, :, 1], "compute")
                else data[cx, cy, :, 1],
            "map": scores
        }

        roi_orientations[tilt][roi_key] = {
            "candidates": candidates,
            "pixel": (cx, cy),
            "best": candidates[0],
            "best_score": float(scores[cx, cy])
        }

        print("TM complete")

In [ ]:
roi_results = {}
roi_scores = {}
roi_orientations = {}

n_keep = 8

for tilt, roi_dict in roi_polar.items():

    roi_results[tilt] = {}
    roi_scores[tilt] = {}
    roi_orientations[tilt] = {}

    for roi_key, signal_pol in roi_dict.items():

        print(f"Tilt {tilt}, ROI {roi_key}")

        res = signal_pol.get_orientation(
            simulation=simulations,
            n_best=n_keep,
            frac_keep=1.0
        )

        roi_results[tilt][roi_key] = res

        # -------------------------
        # scores
        # -------------------------
        data = res.data
        scores = data[..., 0, 1]

        roi_scores[tilt][roi_key] = {
            "mean": float(scores.mean()),
            "max": float(scores.max()),
            "std": float(scores.std()),
            "map": scores
        }

        # -------------------------
        # best orientation
        # -------------------------
        ori_full = res.to_single_phase_orientations()

        cx = ori_full.shape[0] // 2
        cy = ori_full.shape[1] // 2

        best_pixel = (cx, cy)

        ori_best_raw = ori_full[cx, cy, 0]

        ori_best = Orientation(
            ori_best_raw,
            symmetry=GaAs.point_group
        ).map_into_symmetry_reduced_zone()

        roi_orientations[tilt][roi_key] = {
            "best": ori_best,
            "best_pixel": best_pixel,
            "best_score": float(scores[best_pixel])
        }

In [ ]:
res_0_A = roi_polar[-20]["C"]

res_0_A.plot(cmap='viridis_r', norm='log')
res_0_A.add_marker(roi_results[-20]["C"].to_markers(annotate=True))


check one match

# IPF plots

sanity check for template matching plotting the highest scoring pixel.

direction = Vector3d.zvector(). which is the beam direction / zone axis direction

In [ ]:
twin_key = "C"

tilt_keys = sorted(roi_results.keys())  # assumes roi_results[tilt]
n = len(tilt_keys)

fig = plt.figure(figsize=(2*n, 4))

for i, tilt in enumerate(tilt_keys):

    res = roi_results[tilt][twin_key]

    # ori shape: nav_x, nav_y, n_best

    oris = res.to_single_phase_orientations()[:, 0]
    ori = res.to_single_phase_orientations()[6, 7, 0]
    

    ax = fig.add_subplot(1, n, i + 1, projection="ipf", symmetry=GaAs.point_group)

    ax.scatter(
        ori,
        cmap="inferno",
    )

    ax.set_title(f"\nTilt {tilt} | Twin {twin_key}\n")

plt.tight_layout()
plt.show()

In [ ]:
twin_key = "C"
tilt_keys = np.array(sorted(roi_results.keys()))

all_oris = []

for tilt in tilt_keys:
    res = roi_results[tilt][twin_key]
    ori = res.to_single_phase_orientations()[6, 7, 0]
    all_oris.append(ori)

oris_all = Orientation(
    np.stack([o.data.squeeze() for o in all_oris]),
    symmetry=GaAs.point_group
)

cmap = plt.cm.plasma
norm = plt.Normalize(vmin=tilt_keys.min(), vmax=tilt_keys.max())
rgba = cmap(norm(tilt_keys))

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="ipf", symmetry=GaAs.point_group)

ax.scatter(
    oris_all,
    color=rgba,
    alpha=0.9,
    s=80
)

ax.set_title(f"Twin {twin_key}\n")

sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, shrink=0.9)
cbar.set_label("Tilt")
cbar.set_ticks(tilt_keys)
cbar.set_ticklabels([str(t) for t in tilt_keys])

plt.tight_layout()
plt.show()

In [ ]:


# Plot one twin across all tilts
twin_key = "D"
tilt_keys = np.array(sorted(roi_results.keys()))

# data
all_oris = []
all_tilts = []

for tilt in tilt_keys:

    res = roi_results[tilt][twin_key]

    oris = res.to_single_phase_orientations()[4, 4, 0]

    all_oris.append(oris)

    n = oris.data.shape[0]
    all_tilts.append(np.full(n, tilt)) 

oris_all = Orientation(
    np.concatenate([o.data for o in all_oris]),
    symmetry=symmetry.Oh   # m-3m
)

tilt_colors = np.concatenate(all_tilts)

cmap = plt.cm.plasma
norm = plt.Normalize(vmin=tilt_keys.min(), vmax=tilt_keys.max())

# ---- plot ----
fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot(
    111,
    projection="ipf",
    symmetry=symmetry.Oh   # m-3m
)

sc = ax.scatter(
    oris_all,
    c=tilt_colors,
    alpha=0.9,
    cmap=cmap,
    norm=norm,
    s=80
)

ax.set_title(f"Twin {twin_key} \n")

sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, shrink=0.8)
cbar.set_label("Tilt")
cbar.set_ticks(tilt_keys)
cbar.set_ticklabels([str(t) for t in tilt_keys])

plt.tight_layout()
plt.show()

In [ ]:
# misorientaion based on the saved pizel values

import itertools
tilts = sorted(roi_results.keys())

# -------------------------
# ROI labels
# -------------------------
labels = ["A", "B", "C", "D"]

# -------------------------
# all pair combinations
# -------------------------
pairs = list(itertools.combinations(labels, 2))

# -------------------------
# store results
# -------------------------
misorientation_dict = {}

for pair in pairs:

    a, b = pair
    key = f"{a}{b}"

    misorientation_dict[key] = []

    print(f"\n=== {a} vs {b} ===")

    # representative pixels
    px_a, py_a = representative_pixels[a]
    px_b, py_b = representative_pixels[b]

    for tilt in tilts:

        # orientation from representative pixel
        ori_a = roi_results[tilt][a] \
            .to_single_phase_orientations()[px_a, py_a, 0]

        ori_b = roi_results[tilt][b] \
            .to_single_phase_orientations()[px_b, py_b, 0]

        # optional: map to symmetry reduced zone
        ori_a = Orientation(
            ori_a,
            symmetry=GaAs.point_group
        ).map_into_symmetry_reduced_zone()

        ori_b = Orientation(
            ori_b,
            symmetry=GaAs.point_group
        ).map_into_symmetry_reduced_zone()

        # misorientation
        mis = float(
            ori_a.angle_with(
                ori_b,
                degrees=True
            )
        )

        misorientation_dict[key].append(mis)

        print(f"Tilt {tilt:>3}: {mis:.3f}°")

# ==========================================
# Plot
# ==========================================
fig, ax = plt.subplots(figsize=(8, 5))

for key, values in misorientation_dict.items():

    ax.plot(
        tilts,
        values,
        marker="o",
        linewidth=2,
        label=key
    )

# expected twin line
ax.axhline(
    60,
    color="k",
    linestyle="--",
    alpha=0.7,
    label="60° twin"
)

ax.set_xlabel("Tilt")
ax.set_ylabel("Misorientation (degrees)")
ax.set_title("Twin misorientation vs tilt")

ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()

### old

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt

# --- all tilts ---
tilts = sorted(roi_results.keys())

# --- ROI labels ---
labels = ["A", "B", "C", "D"]

# --- all pair combinations ---
pairs = list(itertools.combinations(labels, 2))

# store results
misorientation_dict = {}

for pair in pairs:

    a, b = pair
    key = f"{a}{b}"

    misorientation_dict[key] = []

    print(f"\n=== {a} vs {b} ===")

    for tilt in tilts:

        # best orientation only
        ori_a = roi_results[tilt][a].to_single_phase_orientations()[4, 4, 0]
        ori_b = roi_results[tilt][b].to_single_phase_orientations()[4, 4, 0]

        # misorientation angle
        mis = float(ori_a.angle_with(ori_b, degrees=True))

        misorientation_dict[key].append(mis)

        print(f"Tilt {tilt:>3}: {mis:.3f}°")
# ==========================================
# Plot
# ==========================================
fig, ax = plt.subplots(figsize=(8, 5))

for key, values in misorientation_dict.items():

    ax.plot(
        tilts,
        values,
        marker="o",
        label=key
    )

ax.set_xlabel("Tilt")
ax.set_ylabel("Misorientation (degrees)")
ax.set_title("Twin misorientation vs tilt")

ax.legend()
ax.grid(True)

plt.show()

# stereographic

In [ ]:
tilts, vD = get_twin_vectors_representative("D", candidate_idx=0)

fig = vD.scatter(
    c=tilts,
    cmap="plasma",
    s=120,
    axes_labels=["RD", "TD", None],
    show_hemisphere_label=True,
    return_figure=True,
)

ax = fig.axes[0]

for tilt, v in zip(tilts, vD):
    ax.text(v, s=str(tilt), size=10, offset=(0, 0.04))

ax.set_title("Twin D: stereographic points with tilt labels")
ax.stereographic_grid(True)

fig

In [ ]:
tilts, vD = get_twin_vectors_representative("D", candidate_idx=0)

for i in range(len(tilts) - 1):
    ang = float(vD[i].angle_with(vD[i + 1], degrees=True))
    print(f"{tilts[i]:>3} → {tilts[i+1]:>3}: {ang:.2f}°")

In [ ]:
for idx in range(8):

    ori_m5 = roi_results[-5]["D"].to_single_phase_orientations()[px, py, 0]
    ori_0  = roi_results[0]["D"].to_single_phase_orientations()[px, py, idx]
    ori_5  = roi_results[5]["D"].to_single_phase_orientations()[px, py, 0]

    v_m5 = ori_m5 * Vector3d.zvector()
    v_0  = ori_0  * Vector3d.zvector()
    v_5  = ori_5  * Vector3d.zvector()

    jump_left = float(v_m5.angle_with(v_0, degrees=True))
    jump_right = float(v_0.angle_with(v_5, degrees=True))

    print(
        f"cand {idx}: "
        f"-5→0 = {jump_left:.2f}°, "
        f"0→5 = {jump_right:.2f}°"
    )

In [ ]:
# -------------------------
# C and D using representative pixels
# -------------------------
tilts, vC = get_twin_vectors_same_points("C", candidate_idx=0)
tilts, vD = get_twin_vectors_same_points("D", candidate_idx=0)

arc_pole_C = fit_arc_pole(vC)
arc_pole_D = fit_arc_pole(vD)

fig = vD.scatter(
    c="purple",
    s=90,
    axes_labels=["RD", "TD", None],
    show_hemisphere_label=True,
    return_figure=True,
)

vC.scatter(
    figure=fig,
    c="hotpink",
    s=140,
    edgecolor="black"
)

# reproject poles/circles if they are on the opposite hemisphere
arc_pole_D.scatter(
    figure=fig,
    c="purple",
    marker="*",
    s=250,
    reproject=True
)

arc_pole_C.scatter(
    figure=fig,
    c="hotpink",
    marker="*",
    s=300,
    edgecolor="black",
    reproject=True
)

arc_pole_D.draw_circle(
    figure=fig,
    color="purple",
    linewidth=2,
    reproject=True
)

arc_pole_C.draw_circle(
    figure=fig,
    color="hotpink",
    linewidth=3,
    reproject=True
)

ax = fig.axes[0]
ax.set_title("Twins C and D: stereographic tilt arcs")
ax.stereographic_grid(True)

fig

In [ ]:
print("arc_pole_C:", arc_pole_C.data)
print("arc_pole_D:", arc_pole_D.data)

In [ ]:
# -------------------------
# C and D using representative pixels
# -------------------------
tilts, vC = get_twin_vectors_same_points("C", candidate_idx=0)
tilts, vD = get_twin_vectors_same_points("D", candidate_idx=0)

arc_pole_C = fit_arc_pole(vC)
arc_pole_D = fit_arc_pole(vD)
arc_pole_C_plot = -arc_pole_C

fig = vC.scatter(
    c="hotpink",
    s=90,
    axes_labels=["RD", "TD", None],
    show_hemisphere_label=True,
    return_figure=True,
)

vD.scatter(
    figure=fig,
    c="purple",
    s=90
)

arc_pole_C_plot.scatter(
    figure=fig,
    c="hotpink",
    marker="*",
    s=250
)

 #arc_pole_C.scatter(
    #figure=fig,
    #c="hotpink",
    #marker="*",
    #s=250
#)

arc_pole_D.scatter(
    figure=fig,
    c="purple",
    marker="*",
    s=250
)
arc_pole_C_plot.draw_circle(
    figure=fig,
    color="hotpink",
    linewidth=2
)
""" arc_pole_C.draw_circle(
    figure=fig,
    color="hotpink",
    linewidth=2
) """

arc_pole_D.draw_circle(
    figure=fig,
    color="purple",
    linewidth=2
)

ax = fig.axes[0]
ax.set_title("Twins C and D: stereographic tilt arcs")
ax.stereographic_grid(True)

fig

In [ ]:
def get_twin_vectors_same_points(twin_key, candidate_idx=0):

    px, py = representative_pixels[twin_key]
    tilt_keys = np.array(sorted(roi_results.keys()))

    all_oris = []

    for tilt in tilt_keys:
        ori = roi_results[tilt][twin_key].to_single_phase_orientations()[
            px, py, candidate_idx
        ]

        all_oris.append(ori)

    oris_all = Orientation(
        np.stack([o.data for o in all_oris]),
        symmetry=GaAs.point_group
    )

    vectors = oris_all * Vector3d.zvector()

    return tilt_keys, vectors


def fit_arc_pole(vectors):

    X = vectors.data.reshape(-1, 3)
    X = X / np.linalg.norm(X, axis=1)[:, None]

    _, _, vh = np.linalg.svd(X, full_matrices=False)

    axis_vec = vh[-1]
    axis_vec /= np.linalg.norm(axis_vec)

    return Vector3d(axis_vec)


# -------------------------
# A and B using representative pixels
# -------------------------
tilts, vA = get_twin_vectors_same_points("A", candidate_idx=0)
tilts, vB = get_twin_vectors_same_points("B", candidate_idx=0)

arc_pole_A = fit_arc_pole(vA)
arc_pole_B = fit_arc_pole(vB)

fig = vA.scatter(
    c="red",
    s=90,
    axes_labels=["RD", "TD", None],
    show_hemisphere_label=True,
    return_figure=True,
)

vB.scatter(
    figure=fig,
    c="blue",
    s=90
)

arc_pole_A.scatter(
    figure=fig,
    c="red",
    marker="*",
    s=250
)

arc_pole_B.scatter(
    figure=fig,
    c="blue",
    marker="*",
    s=250
)

arc_pole_A.draw_circle(
    figure=fig,
    color="red",
    linewidth=2
)

arc_pole_B.draw_circle(
    figure=fig,
    color="blue",
    linewidth=2
)

ax = fig.axes[0]
ax.set_title("Twins A and B: stereographic tilt arcs")
ax.stereographic_grid(True)

fig

In [ ]:
angle_AB = float(np.rad2deg(arc_pole_A.angle_with(arc_pole_B)))
angle_AB_small = min(angle_AB, 180 - angle_AB)

print(f"A-B arc-center angle: {angle_AB:.2f}°")
print(f"A-B smallest equivalent angle: {angle_AB_small:.2f}°")

In [ ]:
# -------------------------
# arc-pole angles
# -------------------------

pairs = {
    "AB": (arc_pole_A, arc_pole_B),
    "AC": (arc_pole_A, arc_pole_C),
    "AD": (arc_pole_A, arc_pole_D),
    "BC": (arc_pole_B, arc_pole_C),
    "BD": (arc_pole_B, arc_pole_D),
    "CD": (arc_pole_C, arc_pole_D),
}

for key, (p1, p2) in pairs.items():

    angle = float(
        np.rad2deg(
            p1.angle_with(p2)
        )
    )

    angle_small = min(angle, 180 - angle)

    print(f"{key} arc-center angle: {angle:.2f}°")
    print(f"{key} smallest equivalent angle: {angle_small:.2f}°\n")

In [ ]:
# -------------------------
# C and D using representative pixels
# -------------------------
tilts, vC = get_twin_vectors_same_points("C", candidate_idx=0)
tilts, vD = get_twin_vectors_same_points("D", candidate_idx=0)

arc_pole_C = fit_arc_pole(vC)
arc_pole_D = fit_arc_pole(vD)

fig = vC.scatter(
    c="hotpink",
    s=90,
    axes_labels=["RD", "TD", None],
    show_hemisphere_label=True,
    return_figure=True,
)

vD.scatter(
    figure=fig,
    c="purple",
    s=90
)

arc_pole_C.scatter(
    figure=fig,
    c="hotpink",
    marker="*",
    s=250
)

arc_pole_D.scatter(
    figure=fig,
    c="purple",
    marker="*",
    s=250
)

arc_pole_C.draw_circle(
    figure=fig,
    color="hotpink",
    linewidth=2
)

arc_pole_D.draw_circle(
    figure=fig,
    color="purple",
    linewidth=2
)

ax = fig.axes[0]
ax.set_title("Twins C and D: stereographic tilt arcs")
ax.stereographic_grid(True)

fig

# based on the tutorial

In [ ]:
from orix.vector import Vector3d
from orix.quaternion import Orientation
import numpy as np
import matplotlib.pyplot as plt
from orix import plot  # registers stereographic projection

plt.rcParams["axes.grid"] = True

In [ ]:
def get_twin_vectors(twin_key, px=4, py=4, candidate_idx=0):
    tilt_keys = np.array(sorted(roi_results.keys()))

    vectors = []

    for tilt in tilt_keys:
        ori = roi_results[tilt][twin_key].to_single_phase_orientations()[
            px, py, candidate_idx
        ]

        ori = Orientation(
            ori,
            symmetry=GaAs.point_group
        ).map_into_symmetry_reduced_zone()

        # same direction as IPF-Z / beam direction
        v = ori * Vector3d.zvector()

        vectors.append(v)

    vectors = Vector3d(np.concatenate([v.data for v in vectors]))

    return tilt_keys, vectors

In [ ]:
twin_key = "B"

tilts, vB = get_twin_vectors(
    twin_key,
    px=4,
    py=4,
    candidate_idx=0
)

# fit great-circle pole using all points
X = vB.data.reshape(-1, 3)
X = X / np.linalg.norm(X, axis=1)[:, None]

_, _, vh = np.linalg.svd(X, full_matrices=False)

axis_vec = vh[-1]
axis_vec = axis_vec / np.linalg.norm(axis_vec)

tilt_axis_B = Vector3d(axis_vec)

# plot points
fig = vB.scatter(
    c=tilts,
    cmap="plasma",
    s=120,
    axes_labels=["RD", "TD", None],
    show_hemisphere_label=True,
    return_figure=True,
)

# plot fitted axis / pole
tilt_axis_B.scatter(
    figure=fig,
    c="red",
    marker="*",
    s=250
)

# draw fitted great circle
tilt_axis_B.draw_circle(
    figure=fig,
    color="red",
    linewidth=2
)

fig.axes[0].set_title(f"Twin {twin_key}: stereographic tilt arc + fitted axis")

fig

In [ ]:
tilts, vA = get_twin_vectors("A", px=4, py=4, candidate_idx=0)
tilts, vB = get_twin_vectors("B", px=4, py=4, candidate_idx=0)

arc_pole_A = vA[0].cross(vA[-1]).unit
arc_pole_B = vB[0].cross(vB[-1]).unit

fig = vA.scatter(
    c="red",
    s=90,
    axes_labels=["RD", "TD", None],
    show_hemisphere_label=True,
    return_figure=True,
)

vB.scatter(
    figure=fig,
    c="blue",
    s=90
)

arc_pole_A.scatter(
    figure=fig,
    c="red",
    marker="*",
    s=250
)

arc_pole_B.scatter(
    figure=fig,
    c="blue",
    marker="*",
    s=250
)

arc_pole_A.draw_circle(
    figure=fig,
    color="red",
    linewidth=2
)

arc_pole_B.draw_circle(
    figure=fig,
    color="blue",
    linewidth=2
)

fig

In [ ]:
angle_AB = float(np.rad2deg(arc_pole_A.angle_with(arc_pole_B)))
angle_AB_small = min(angle_AB, 180 - angle_AB)

print(f"A-B arc-center angle: {angle_AB:.2f}°")
print(f"A-B smallest equivalent angle: {angle_AB_small:.2f}°")

In [ ]:


# ==========================================
# SAME orientations as your working IPF plot
# ==========================================

twin_key = "B"
tilt_keys = np.array(sorted(roi_results.keys()))

all_oris = []
all_tilts = []

for tilt in tilt_keys:

    res = roi_results[tilt][twin_key]

    # EXACT same orientation as your good IPF
    ori = res.to_single_phase_orientations()[4, 4, 0]

    all_oris.append(ori)
    all_tilts.append(tilt)

# stack orientations
oris_all = Orientation(
    np.stack([o.data for o in all_oris]),
    symmetry=GaAs.point_group
)

# ==========================================
# Convert orientations -> stereographic vectors
# ==========================================

vectors = oris_all * Vector3d.zvector()

# ==========================================
# Fit great-circle pole using SVD
# ==========================================

X = vectors.data.reshape(-1, 3)
X = X / np.linalg.norm(X, axis=1)[:, None]

_, _, vh = np.linalg.svd(X, full_matrices=False)

axis_vec = vh[-1]
axis_vec /= np.linalg.norm(axis_vec)

tilt_axis = Vector3d(axis_vec)

# ==========================================
# Plot stereographic projection
# ==========================================

fig = vectors.scatter(
    c=all_tilts,
    cmap="plasma",
    s=120,
    axes_labels=["RD", "TD", None],
    show_hemisphere_label=True,
    return_figure=True,
)

# draw trajectory between points
ax = fig.axes[0]
ax.plot(vectors, color="black", linewidth=1)

# plot fitted tilt-axis pole
tilt_axis.scatter(
    figure=fig,
    c="red",
    marker="*",
    s=250
)

# draw fitted great circle
tilt_axis.draw_circle(
    figure=fig,
    color="red",
    linewidth=2
)

ax.set_title(
    f"Twin {twin_key}: Stereographic tilt trajectory"
)

fig.axes[0].stereographic_grid(True)

plt.show()

print("Tilt-axis pole:")
print(tilt_axis.data)

# for two twins

In [ ]:
def get_twin_vectors_same_points(twin_key, px=4, py=4, candidate_idx=0):
    px, py = representative_pixels[twin_key]

    tilt_keys = np.array(sorted(roi_results.keys()))

    all_oris = []

    for tilt in tilt_keys:
        ori = roi_results[tilt][twin_key].to_single_phase_orientations()[
            px, py, candidate_idx
        ]

        all_oris.append(ori)

    oris_all = Orientation(
        np.stack([o.data for o in all_oris]),
        symmetry=GaAs.point_group
    )

    vectors = oris_all * Vector3d.zvector()

    return tilt_keys, vectors


def fit_arc_pole(vectors):

    X = vectors.data.reshape(-1, 3)
    X = X / np.linalg.norm(X, axis=1)[:, None]

    _, _, vh = np.linalg.svd(X, full_matrices=False)

    axis_vec = vh[-1]
    axis_vec /= np.linalg.norm(axis_vec)

    return Vector3d(axis_vec)




tilts, vA = get_twin_vectors_same_points("C", px=4, py=4, candidate_idx=0)
tilts, vB = get_twin_vectors_same_points("D", px=4, py=4, candidate_idx=0)

arc_pole_A = fit_arc_pole(vA)
arc_pole_B = fit_arc_pole(vB)

fig = vA.scatter(
    c="hotpink",
    s=90,
    axes_labels=["RD", "TD", None],
    show_hemisphere_label=True,
    return_figure=True,
)

vB.scatter(
    figure=fig,
    c="purple",
    s=90
)

arc_pole_A.scatter(
    figure=fig,
    c="hotpink",
    marker="*",
    s=250
)

arc_pole_B.scatter(
    figure=fig,
    c="purple",
    marker="*",
    s=250
)

arc_pole_A.draw_circle(
    figure=fig,
    color="hotpink",
    linewidth=2
)

arc_pole_B.draw_circle(
    figure=fig,
    color="purple",
    linewidth=2
)

ax = fig.axes[0]
ax.set_title("Twins C and D: stereographic tilt arcs")
ax.stereographic_grid(True)

fig

## measure the angle between the fitted arc centers

In [ ]:
angle_AB = float(np.rad2deg(arc_pole_A.angle_with(arc_pole_B)))
angle_AB_small = min(angle_AB, 180 - angle_AB)

print(f"A-B arc-center angle: {angle_AB:.2f}°")
print(f"A-B smallest equivalent angle: {angle_AB_small:.2f}°")

In [ ]:
for tilt in sorted(roi_results.keys()):

    ori_A = roi_results[tilt]["A"].to_single_phase_orientations()[4, 4, 0]
    ori_B = roi_results[tilt]["B"].to_single_phase_orientations()[4, 4, 0]

    ori_A = Orientation(ori_A, symmetry=GaAs.point_group).map_into_symmetry_reduced_zone()
    ori_B = Orientation(ori_B, symmetry=GaAs.point_group).map_into_symmetry_reduced_zone()

    mis = ori_A.angle_with(ori_B, degrees=True)

    print(f"Tilt {tilt:>3}: A-B misorientation = {mis:.2f}°")

## represent the crystals in the same frame

In [ ]:
def get_oris_representative(twin_key, candidate_idx=0):

    px, py = representative_pixels[twin_key]
    tilt_keys = np.array(sorted(roi_results.keys()))

    all_oris = []

    for tilt in tilt_keys:
        ori = roi_results[tilt][twin_key].to_single_phase_orientations()[
            px, py, candidate_idx
        ]
        all_oris.append(ori)

    oris = Orientation(
        np.stack([o.data for o in all_oris]),
        symmetry=GaAs.point_group
    )

    return tilt_keys, oris

In [ ]:
tilts, oris_A = get_oris_representative("A")
tilts, oris_B = get_oris_representative("B")

# choose reference tilt, e.g. 0 degrees
ref_tilt = 0
i_ref = list(tilts).index(ref_tilt)

ori_A_ref = oris_A[i_ref]
ori_B_ref = oris_B[i_ref]

# relative rotation bringing B reference into A reference
g_rel_B_to_A = ori_A_ref * ~ori_B_ref

# express all B orientations in A frame
oris_B_in_A = g_rel_B_to_A * oris_B

# now make stereographic vectors in same frame
vA = oris_A * Vector3d.zvector()
vB_in_A = oris_B_in_A * Vector3d.zvector()

arc_pole_A = fit_arc_pole(vA)
arc_pole_B_in_A = fit_arc_pole(vB_in_A)

In [ ]:
fig = vA.scatter(
    c="red",
    s=90,
    axes_labels=["RD", "TD", None],
    show_hemisphere_label=True,
    return_figure=True,
)

vB_in_A.scatter(
    figure=fig,
    c="blue",
    s=90
)

arc_pole_A.scatter(
    figure=fig,
    c="red",
    marker="*",
    s=250
)

arc_pole_B_in_A.scatter(
    figure=fig,
    c="blue",
    marker="*",
    s=250
)

arc_pole_A.draw_circle(
    figure=fig,
    color="red",
    linewidth=2
)

arc_pole_B_in_A.draw_circle(
    figure=fig,
    color="blue",
    linewidth=2
)

ax = fig.axes[0]
ax.set_title("A and B arcs with B transformed into A reference frame")
ax.stereographic_grid(True)

fig

In [ ]:
angle_AB_same_frame = float(
    arc_pole_A.angle_with(
        arc_pole_B_in_A,
        degrees=True
    )
)

angle_AB_same_frame_small = min(
    angle_AB_same_frame,
    180 - angle_AB_same_frame
)

print(f"A-B arc-pole angle, same frame: {angle_AB_same_frame:.2f}°")
print(f"A-B smallest equivalent: {angle_AB_same_frame_small:.2f}°")